In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("use catalog `ecom_azure_project`")
spark.sql("create schema if not exists silver_schema")

silver_run_id = str(uuid.uuid4())
print("Current Silver Run ID: ", silver_run_id)

### Step 2: Silver control table

This table stores the latest silver processing state for each entity.

it helps us track:
- the latest Bronze run already processed for silver
- the latest bronze ingestion timestamp already processed
- how many rows were merged in the latest silver run

In [0]:
spark.sql("""
          create table if not exists ecom_azure_project.silver_schema.processing_control(
              layer string,
              entity_name string,
              last_processed_bronze_run_id string,
              last_processed_bronze_ingested_at timestamp,
              rows_merged bigint,
              run_status string,
              silver_run_id string,
              updated_at timestamp
          )
          using delta
          
          
          
          
          """)

### Step 3 - Helper functions

this cell contains reusable logic for silver:
- upsert_to_silver() merges cleaned/transformed rows into the silver target table
- get_last_processsed_bronze_ingested_at() reads the silver watermark
- upsert_silver_control() updates the silver control table
- get_incremental_bronze() reads only new bronze rows that silver has not processed yet.


In [0]:
def upsert_to_silver(df_source, target_table, join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        (dt.alias("target")
        .merge(df_source.alias("source"), f"target.{join_key} = source.{join_key}")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    else:
        df_source.write.format("delta").saveAsTable(target_table)


       

In [0]:
def get_last_processsed_bronze_ingested_at(entity_name: str):
    ctrl = (
        spark.table("ecom_azure_project.silver_schema.processing_control")
        .filter(
            (F.col("layer") == "silver") &
            (F.col("entity_name") == entity_name)&
            (F.col("run_status") == "SUCCESS")
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
       
    )
    rows = ctrl.collect()
    if not rows:
        return None
    
    return rows[0]["last_processed_bronze_ingested_at"]
 
    
    

In [0]:
def upsert_silver_control(entity_name, last_processed_bronze_run_id, last_processed_bronze_ingested_at, rows_merged):
    ctrl_df = spark.createDataFrame(
        [
            (
                "silver",
                entity_name,
                last_processed_bronze_run_id,
                last_processed_bronze_ingested_at,
                int(rows_merged),
                "SUCCESS",
                silver_run_id,
                datetime.utcnow(),
            )
        ],
        schema="""
        layer string,
        entity_name string,
        last_processed_bronze_run_id string,
        last_processed_bronze_ingested_at timestamp,
        rows_merged bigint,
        run_status string,
        silver_run_id string,
        
        
        
        """
    )
    dt = DeltaTable.forName(spark, "ecom_azure_project.silver_schema.processing_control")
    (dt.alias("t")
        .merge(ctrl_df.alias("s"), "t.layer = s.layer AND t.entity_name = s.entity_name")
        .whenMatchedUpdate(set={
            "last_processed_bronze_run_id": "s.last_processed_bronze_run_id",
            "last_processed_bronze_ingested_at": "s.last_processed_bronze_ingested_at",
            "rows_merged": "s.rows_merged",
            "run_status": "s.run_status",
            "silver_run_id": "s.silver_run_id",
            "updated_at": "s.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
     
     
     )
    
    


In [0]:
def get_incremental_bronze(bronze_table, entity_name):
    last_ingested_at = get_last_processsed_bronze_ingested_at(entity_name)
    bronze_df = spark.table(bronze_table)
    if last_ingested_at is None:
        return bronze_df, last_ingested_at
    

    return bronze_df.filter(F.col("bronze_ingested_at") > F.lit(last_ingested_at)), last_ingested_at
    

### Step 4 - Orders Incremental processing

This cell process orders from bronze to silver.

it does following:
- reads only new bronze order rows
- cleans values like order_status and order_amount
- keeps only the latest version per order_id
- validates business rules
- sends bad rows to quarantine
- merges good rows into orders_transformed

In [0]:
# Read only the bronze order rows that silver has not processed
orders_inc, last_orders_ingested_at = get_incremental_bronze(
    "ecom_azure_project.bronze_schema.orders_raw",
    "orders"
)

# Count the incremental order rows entering Silver in this run
orders_inc_count = orders_inc.count()
print(f"orders rows_to_process_in_silver : {orders_inc_count}")

# Only run Silver order cleaning and validation when there are new bronze order rows.
if orders_inc_count > 0:
    # create a window that kepps the latest order record for each order_id.
    order_window = Window.partitionBy("order_id").orderBy(
        F.col("updated_at").cast("timestamp").desc(),
        F.col("bronze_ingested_at").desc()
    )

    # Start the silver order-cleaning pipeline. This block standardized and deduplications raw order records.

    orders_cleaned = (
        orders_inc
        # Standardize order_status to uppercase so values such as shipped and SHIPPED become consistent.
        .withColumn("order_status", F.upper(F.trim(F.col("order_status"))))
        .withColumn("order_status", F.when(F.col("order_status") == "", F.lit(None)).otherwise(F.col("order_status")))
        # Remove fromatting characters from order_amount so it can cast to a numeric type.
        .withColumn("order_amount", F.regexp_replace(F.col("order_amount"), r"[$,]", ""))
        .withColumn("order_amount", F.when(F.trim(F.col("order_amount")).isin("N/A", "NULL", "??", ""), None).otherwise(F.col("order_amount")))
        .withColumn("order_amount", F.col("order_amount").cast("double"))
        .withColumn("created_at", F.to_timestamp(F.col("created_at"))
        .withColumn("updated_at", F.to_timestamp(F.col("updated_at"))
        # Assign a row number inside each business key so we can keep only the latest version of that record.
        .withColumn("row_rank", F.row_number().over(order_window))
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )
    

